In [3]:
#YOLOv8 Classification
#pip install ultralytics

from ultralytics import YOLO

# Load a model
model = YOLO("yolov8n.pt")  # load a pretrained model (recommended for training)

# Use the model
results = model("https://ultralytics.com/images/bus.jpg" , save=True)  # predict on an image


Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
image 1/1 /home/vincent/Code/Computer Vision NDHU/1217/bus.jpg: 640x480 4 persons, 1 bus, 1 stop sign, 125.6ms
Speed: 2.9ms preprocess, 125.6ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 480)
Results saved to /home/vincent/Code/Computer Vision NDHU/1217/runs/detect/predict2


In [2]:
#YOLOv8 Classification
#pip install ultralytics

from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO('yolov8n.pt')
cap = cv2.VideoCapture(0)
#cap.set(3, 640)
#cap.set(4, 480)

while True:
    ret, img = cap.read()
    if ret == False:
        break
    img = cv2.flip(img, 1)
    # BGR to RGB conversion is performed under the hood
    # see: https://github.com/ultralytics/ultralytics/issues/2575
    results = model.predict(img, conf=0.25)  # Set your desired confidence threshold, default = 0.25

    for result in results:
        for box in result.boxes:
            #print(box.xyxy) #box.xyxy.cpu()
            left, top, right, bottom = np.array(box.xyxy, dtype=np.uint16).squeeze() #convert from tensor to list
            #print(left, top, right, bottom)
            width = right - left
            height = bottom - top
            center = (left + int((right-left)/2), top + int((bottom-top)/2))
            label = results[0].names[int(box.cls)]
            confidence = float(box.conf.cpu())

            cv2.rectangle(img, (left, top),(right, bottom), (255, 0, 0), 2)
            cv2.putText(img, label+' '+'{:.2f}'.format(confidence),(left+5, bottom-10),cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 1, cv2.LINE_AA)
        
    cv2.imshow('YOLO V8 Detection', img)     
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


0: 480x640 1 person, 1 wine glass, 1 chair, 1 couch, 1 tv, 119.6ms
Speed: 4.8ms preprocess, 119.6ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 wine glass, 1 chair, 1 couch, 1 tv, 128.5ms
Speed: 2.8ms preprocess, 128.5ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 wine glass, 1 cup, 1 chair, 1 couch, 1 tv, 112.6ms
Speed: 1.9ms preprocess, 112.6ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 wine glass, 1 chair, 1 couch, 1 tv, 102.1ms
Speed: 9.4ms preprocess, 102.1ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 wine glass, 1 chair, 1 couch, 1 tv, 103.3ms
Speed: 3.0ms preprocess, 103.3ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 wine glass, 1 cup, 1 chair, 1 tv, 102.7ms
Speed: 3.1ms preprocess, 102.7ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

## Practice : People detector and analyzer
1. Input images from wiiplay.mp4 with frame number between 41000 and 44000.
2. Use YOLOv8 to detect people, mark as red rectangle, and count how many persons in each frame. (hint: check label == 'person')
3. Try to find out which frame contains the most number of persons. (print the number of persons on the upper-left corner)
4. (optional) Try to find out which frame containes the largest person. (print the size of its bounding box on the upper-left corner)
5. (optional) Try to find out which frame containes the smallest person. (print the size of its bounding box on the upper-left corner)
6. Show the three output frames you found.
7. Verify the correctness of your output, then adjust the desired confidence threshold for improvement. 
8. Upload your Jupyter code file (*.ipynb)

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

cap = cv2.VideoCapture('WiiPlay.mp4')
# Check if the video file is opened correctly
if not cap.isOpened():
    raise IOError("Cannot open the video file")

s_frame_seq = 41000
f_frame_seq = 44000
out_size = (640, 360)

cap.set(cv2.CAP_PROP_POS_FRAMES , s_frame_seq);
ret, frame = cap.read()
cur_frame = cv2.resize(frame, out_size, 0, 0, interpolation=cv2.INTER_AREA)

max_c = 0

count = 0


while True:
    ret, frame = cap.read()
    if ret == False:
        break
        
    current_frame_number = cap.get(cv2.CAP_PROP_POS_FRAMES)
    if current_frame_number == f_frame_seq:
        break
        
    cur_frame = cv2.resize(frame, out_size, 0, 0, interpolation=cv2.INTER_AREA)
    results = model.predict(cur_frame, conf=0.20)

    count = 0
    
    for result in results:
        for box in result.boxes:
            print(box.xyxy) #box.xyxy.cpu()
            left, top, right, bottom = np.array(box.xyxy, dtype=np.uint16).squeeze() #convert from tensor to list
            print(left, top, right, bottom)
            width = right - left
            height = bottom - top
            center = (left + int((right-left)/2), top + int((bottom-top)/2))
            label = results[0].names[int(box.cls)]
            confidence = float(box.conf.cpu())
            if label == 'person':
                count += 1
                cv2.rectangle(cur_frame, (left, top),(right, bottom), (0, 0, 255), 2)
                #cv2.putText(cur_frame, label+' '+'{:.2f}'.format(confidence),(left+5, bottom-10),cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 1, cv2.LINE_AA)

    if count > max_c:
        max_c = count
        max_frame = cur_frame
        cv2.putText(max_frame, str(max_c), (10, 350), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
        
    cv2.imshow('WillPlay Detection', cur_frame)
    cv2.imshow('Most person',max_frame)
    
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


0: 384x640 11 persons, 53.4ms
Speed: 1.8ms preprocess, 53.4ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
tensor([[470.2916, 251.7883, 536.6721, 350.1473]])
470 251 536 350
tensor([[295.8176, 127.7763, 337.2589, 205.3366]])
295 127 337 205
tensor([[140.4622, 295.7339, 178.6939, 349.6201]])
140 295 178 349
tensor([[315.6908, 165.2650, 355.0828, 257.4948]])
315 165 355 257
tensor([[147.0174, 144.5386, 194.3117, 247.6434]])
147 144 194 247
tensor([[440.3278,   2.4081, 473.5987,  77.3160]])
440 2 473 77
tensor([[290.1473, 248.0457, 352.1588, 349.6707]])
290 248 352 349
tensor([[469.2623, 145.7636, 520.1547, 246.0471]])
469 145 520 246
tensor([[281.5919, 201.9868, 322.5076, 298.3044]])
281 201 322 298
tensor([[469.0349,  11.0442, 506.9420,  98.0108]])
469 11 506 98
tensor([[450.9740,  39.3979, 498.8949, 144.6729]])
450 39 498 144

0: 384x640 13 persons, 56.1ms
Speed: 1.7ms preprocess, 56.1ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)
tensor([[140.